# BellaSDK: Google Colab 3D Asset Generator (Microsoft TRELLIS)

This notebook connects your local **Antigravity IDE** to a Google Colab GPU runtime to generate game-ready 3D models and pack Godot 4 ORM textures.

### How to run inside Antigravity IDE (GUI):
1. In the top-right corner of this notebook editor, click **Select Kernel**.
2. Choose **Colab** -> Sign in with your Google account.
3. Select your desired runtime (**T4 GPU** or **A100 GPU**).
4. Execute the cells sequentially.

### Step 1: Verify Remote GPU Runtime

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU is not attached! Go to Kernel -> Change Runtime to T4/A100 GPU.")

### Step 2: Install Dependencies
Installs lightweight client and geometry tools. Avoids Python 3.13 CUDA compilation issues.

In [ ]:
!pip install -q --upgrade pip
!pip install -q gradio_client trimesh pillow numpy scipy

### Step 3: Gate 1 — Asset Specifications & Invariants

In [ ]:
# Configure your asset metadata here
PROP_SLUG = "industrial_pipe_valve"
TARGET_LAYER = "Layer 3 Interactive"
TARGET_TRI_BUDGET = 1200
METALLIC_VALUE = 0.85  # Cast iron flange + brass lock nut

print(f"Asset Locked: '{PROP_SLUG}' | Layer: {TARGET_LAYER} | Tri Limit: {TARGET_TRI_BUDGET} | Metallic: {METALLIC_VALUE}")

### Step 4: Connect to TRELLIS Remote Pipeline
Connects to hosted TRELLIS endpoint via gradio_client (bypasses Colab Python 3.13 wheel issues).

In [ ]:
from gradio_client import Client

# Primary: Active community mirror on ZeroGPU; Fallback: Microsoft official
spaces = ['trellis-community/TRELLIS', 'microsoft/TRELLIS.2']
client = None
for space_name in spaces:
    try:
        print(f'Attempting connection to {space_name}...')
        client = Client(space_name)
        print(f'Connected successfully to: {space_name}')
        break
    except Exception as e:
        print(f'Space {space_name} unavailable: {e}')

assert client is not None, 'Could not connect to any active TRELLIS Hugging Face space.'
client.view_api()

### Step 5: Gate 2 & 3 — Upload Concept Plate & Run Inference
Upload your 2D orthographic concept plate (neutral gray background, flat studio lighting) to `/content/input.png` or specify an image URL.

In [ ]:
import os
import shutil
import trimesh
from gradio_client import handle_file

INPUT_IMAGE_PATH = '/content/input.png'
assert os.path.exists(INPUT_IMAGE_PATH), f'Upload concept image to {INPUT_IMAGE_PATH} before running!'

OUTPUT_DIR = f'/content/output_{PROP_SLUG}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
glb_target_path = f'{OUTPUT_DIR}/{PROP_SLUG}.glb'

# 1. Initialize server session
try:
    client.predict(api_name='/start_session')
except Exception:
    pass

# 2. Preprocess plate (background removal / silhouette isolation)
print(f'Preprocessing {INPUT_IMAGE_PATH} via /preprocess_image...')
prep_out = client.predict(
    image=handle_file(INPUT_IMAGE_PATH),
    api_name='/preprocess_image'
)
prep_file = prep_out['path'] if isinstance(prep_out, dict) and 'path' in prep_out else prep_out
print(f'Preprocessed plate ready at: {prep_file}')

# 3. Generate 3D representations & extract GLB
print('Generating 3D model via /generate_and_extract_glb (takes ~30-60s on ZeroGPU)...')
video_preview, viewer_glb, download_glb = client.predict(
    image=handle_file(prep_file),
    multiimages=[],
    seed=42,
    ss_guidance_strength=7.5,
    ss_sampling_steps=12,
    slat_guidance_strength=3.0,
    slat_sampling_steps=12,
    multiimage_algo='stochastic',
    mesh_simplify=0.95,
    texture_size=1024,
    api_name='/generate_and_extract_glb'
)

glb_src = download_glb['path'] if isinstance(download_glb, dict) and 'path' in download_glb else download_glb
shutil.copy(glb_src, glb_target_path)
print(f'Downloaded raw GLB to: {glb_target_path}')

# 4. Decimate to target budget (Step 6)
loaded = trimesh.load(glb_target_path, force='mesh')
tri_count = len(loaded.faces)
print(f'Raw mesh faces: {tri_count}')
if tri_count > TARGET_TRI_BUDGET:
    print(f'Decimating mesh to target budget: {TARGET_TRI_BUDGET}...')
    decimated = loaded.simplify_quadric_decimation(TARGET_TRI_BUDGET)
    decimated.export(glb_target_path)
    tri_count = len(decimated.faces)
    print(f'Decimated mesh faces: {tri_count}')
print(f'[SUCCESS] GLB ready at: {glb_target_path}')

### Step 6: Gate 4 — Pack Godot 4 ORM Texture
Packs **Ambient Occlusion (R)**, **Roughness (G)**, and **Metallic (B)** into a single 3-channel PNG to cut GPU texture samplers.

In [ ]:
import numpy as np
from PIL import Image

def pack_orm(ao_path, rough_path, metal_val, out_path, size=(2048, 2048)):
    ao_img = Image.open(ao_path).convert("L").resize(size) if ao_path and os.path.exists(ao_path) else Image.new("L", size, 255)
    rough_img = Image.open(rough_path).convert("L").resize(size) if rough_path and os.path.exists(rough_path) else Image.new("L", size, 190)
    metal_int = int(np.clip(metal_val * 255.0, 0, 255))
    metal_img = Image.new("L", size, metal_int)
    
    orm = Image.merge("RGB", (ao_img, rough_img, metal_img))
    orm.save(out_path, format="PNG", optimize=True)
    print(f"Packed Godot 4 ORM texture saved: {out_path} ({size[0]}x{size[1]})")
    return out_path

orm_path = f"{OUTPUT_DIR}/{PROP_SLUG}_orm.png"
pack_orm(None, None, METALLIC_VALUE, orm_path)

### Step 7: Package & Send to Local BellaSDK
Bundles the GLB and textures into a JSON payload for local ingest.

In [ ]:
import json
import base64

bundle = {
    "prop_slug": PROP_SLUG,
    "target_layer": TARGET_LAYER,
    "tri_count": tri_count if 'tri_count' in locals() else TARGET_TRI_BUDGET,
    "generator": "TRELLIS",
    "files": {}
}

for fname in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath) and fname.endswith(('.glb', '.png', '.tres', '.json')):
        with open(fpath, "rb") as f:
            bundle["files"][fname] = base64.b64encode(f.read()).decode("utf-8")

bundle_json_path = f"/content/{PROP_SLUG}_bundle.json"
with open(bundle_json_path, "w", encoding="utf-8") as f:
    json.dump(bundle, f)

print(f"[SUCCESS] Asset bundle ready: {bundle_json_path}")
print(f"Files in bundle: {list(bundle['files'].keys())}")

# Automatic browser download when connected to Colab
try:
    from google.colab import files
    files.download(bundle_json_path)
    print("Triggered bundle download to local machine.")
except Exception:
    pass

print("\nOn your local Antigravity terminal, ingest the asset directly by running:")
print(f"python3 tools/scripts/colab_asset_sync.py ingest <path_to_downloaded_bundle>")